# Deber — Semana 3: Carga de Datos, Series/DataFrames, loc/iloc y Análisis de Texto
## Análisis de Datos (TDSD353) | Período 2026-A
## ESFOT — Escuela Politécnica Nacional


**Nombre:** Danny Alexander Salcedo Vasquez
**Fecha:** 27/04/2026
<br/>
---

### Objetivo de este Taller
Aplicar las técnicas aprendidas en clase para:
1. Cargar datos desde archivos **JSON**, **TXT** (CSV) y **XML** usando Pandas.
2. Detectar y documentar **problemas de calidad de datos**.
3. Unificar múltiples DataFrames con `pd.concat()` y exportar a CSV.
4. Comprender y usar **Series** y **DataFrames**.
5. Acceder a datos con **`loc`** (por etiqueta) e **`iloc`** (por posición).
6. Descargar y procesar texto desde la web con `urllib`.

### Archivos de trabajo
| Archivo | Contenido | IDs |
|---------|-----------|-----|
| `productos.json` | 20 productos de librería | 1–20 |
| `productos.txt` | 20 productos de librería (formato CSV) | 21–40 |
| `productos.xml` | 20 productos de librería | 41–60 |
| `netflix.csv` | Catálogo de Netflix (~8800 títulos) | — |

> ⚠️ Los archivos de productos contienen **errores intencionales** (`"error"`, `"N/A"`, `null`, stock negativo, campos vacíos) para ilustrar problemas reales de calidad de datos.

---

---
## PARTE A: Pipeline de datos — Inventario de Papelería

Tenemos 3 archivos con datos del inventario de una Papelería:
- `productos.json` — 20 productos (IDs 1–20)
- `productos.txt` — 20 productos (IDs 21–40, separados por coma)
- `productos.xml` — 20 productos (IDs 41–60)

Columnas compartidas: `id`, `nombre`, `precio_compra`, `categoria`, `stock`, `proveedor`, `precio_venta_publico`

> ⚠️ Los datos contienen errores intencionales (`"error"`, `"N/A"`, `null`, stock negativo, campos vacíos) para ilustrar problemas reales de calidad de datos.

### A-1. Importar librerías

| Librería | Uso |
|----------|-----|
| `pandas` | Manipulación de datos tabulares |
| `urllib.request` | Descarga de archivos desde la web |
| `collections.Counter` | Conteo de frecuencias |
| `time` | Medición de tiempos |
| `re` | Expresiones regulares para limpieza de texto |

In [1402]:
!pip install lxml
import pandas as pd
import urllib.request
from collections import Counter
import time as time
import re


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### A-2. Cargar JSON con `pd.read_json()`

**JSON** (JavaScript Object Notation): formato ligero muy común en APIs web.

```python
df = pd.read_json('archivo.json')
```

Parámetros útiles: `orient` (formato del JSON), `encoding`, `dtype` (forzar tipos).

In [1403]:
df1=pd.read_json("productos.json")
df1

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,1,Bolígrafo BIC Azul,0.35,Escritura,500,DistribuPapel S.A.,0.75
1,2,Cuaderno Universitario 100 hojas,1.8,Cuadernos,300,Norma Ecuador,3.50
2,3,Lápiz HB Faber-Castell,0.2,Escritura,800,DistribuPapel S.A.,0.45
3,4,Resaltador Amarillo,error,Escritura,150,OfficeMax Ecuador,1.20
4,5,Carpeta Manila A4,0.15,Archivo,1000,PapelPro Cía. Ltda.,0.30
5,6,Tijeras Escolares 13cm,0.9,Manualidades,200,OfficeMax Ecuador,1.80
6,7,Corrector Líquido 18ml,None,Escritura,180,Norma Ecuador,1.50
7,8,Goma de Borrar Blanca,0.25,Escritura,600,DistribuPapel S.A.,0.55
8,9,Resma Papel Bond A4 75g,3.2,Papel,250,PapelPro Cía. Ltda.,5.50
9,10,Marcador Permanente Negro,0.6,Escritura,400,OfficeMax Ecuador,1.10


### A-3. Cargar TXT con `pd.read_csv()`

**TXT tabular** (datos separados por coma): los archivos `.txt` con formato de tabla se leen con `read_csv()` ajustando el separador.

```python
df = pd.read_csv('archivo.txt', sep=',')   # separador: coma
df = pd.read_csv('archivo.txt', sep='\t')  # separador: tabulación
```

> **Diferencia CSV vs TXT**: ambos son texto plano. La extensión `.txt` no implica un formato fijo — siempre revisa el separador antes de cargar.

In [1404]:
df2=pd.read_csv('productos.txt', sep=',')
df2

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,21,Sacapuntas Doble Uso,0.15,Escritura,700,DistribuPapel S.A.,0.35
1,22,Cuaderno Espiral A5 80 hojas,1.40,Cuadernos,220,Norma Ecuador,2.80
2,23,Lápiz de Color x24,2.10,Manualidades,160,Norma Ecuador,4.20
3,24,Pegamento en Barra 40g,0.55,Manualidades,430,OfficeMax Ecuador,1.05
4,25,Papel Lustre x10 colores,0.60,Papel,380,PapelPro Cía. Ltda.,1.20
5,26,Borrador Tinta-Lápiz,0.30,Escritura,550,DistribuPapel S.A.,0.65
6,27,Bolígrafo Gel Negro 0.5mm,NaN,Escritura,310,OfficeMax Ecuador,1.40
7,28,Agenda 2025 Ejecutiva,5.20,Organización,85,PapelPro Cía. Ltda.,9.80
8,29,Scotch Doble Faz 12mm,NaN,Manualidades,265,OfficeMax Ecuador,1.60
9,30,Perforadora 2 huecos,4.50,Organización,60,DistribuPapel S.A.,8.50


### A-4. Cargar XML con `pd.read_xml()`

**XML** (eXtensible Markup Language): formato jerárquico usado en servicios web SOAP y datos gubernamentales.

```python
df = pd.read_xml('archivo.xml', xpath='.//producto')
```

El parámetro `xpath` indica qué nodos representan cada fila. 

In [1405]:
df3=pd.read_xml('productos.xml', xpath='.//producto')
df3

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,41,Tiza Blanca Caja x100,1.50,Escritura,180,Norma Ecuador,3.00
1,42,Papelógrafo Cuadriculado x50,2.80,Papel,130,PapelPro Cía. Ltda.,5.20
2,43,Organizador de Escritorio Plástico,3.60,Organización,65,OfficeMax Ecuador,6.80
3,44,Cuaderno Cosido 50 hojas,0.90,Cuadernos,texto,Norma Ecuador,1.80
4,45,Cartulina de Colores x10,0.70,Papel,320,PapelPro Cía. Ltda.,1.40
5,46,Borrador Nata Pelikan,NaN,Escritura,670,DistribuPapel S.A.,0.60
6,47,Mochila Escolar Mediana,12.50,Accesorios,50,NaN,24.00
7,48,Plumones Lavables x10,1.80,Manualidades,275,Norma Ecuador,3.50
8,49,Cuaderno Universitario Cuadriculado,1.80,Cuadernos,190,Norma Ecuador,error
9,50,Tijeras para Zurdo 21cm,1.40,Manualidades,80,OfficeMax Ecuador,2.70


### A-5. Verificar consistencia y detectar errores

Antes de concatenar, verificamos que los 3 DataFrames tengan las **mismas columnas** y documentamos los problemas de calidad.

Errores comunes en datos reales:
- Valores `null` / `NaN` (datos faltantes)
- Texto donde debería haber números (`"error"`, `"N/A"`, `"texto"`)
- Valores lógicamente inválidos (Ejemplo stock negativo)
- Campos vacíos (`""`)

In [1406]:
print(df1.columns)
print(df2.columns)
print(df3.columns)

Index(['id', 'nombre', 'precio_compra', 'categoria', 'stock', 'proveedor',
       'precio_venta_publico'],
      dtype='str')
Index(['id', 'nombre', 'precio_compra', 'categoria', 'stock', 'proveedor',
       'precio_venta_publico'],
      dtype='str')
Index(['id', 'nombre', 'precio_compra', 'categoria', 'stock', 'proveedor',
       'precio_venta_publico'],
      dtype='str')


In [1407]:
print("Nulos df1:\n", df1.isnull().sum())
print("Nulos df2:\n", df2.isnull().sum())
print("Nulos df3:\n", df3.isnull().sum())

Nulos df1:
 id                      0
nombre                  0
precio_compra           1
categoria               0
stock                   0
proveedor               1
precio_venta_publico    0
dtype: int64
Nulos df2:
 id                      0
nombre                  0
precio_compra           2
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64
Nulos df3:
 id                      0
nombre                  0
precio_compra           2
categoria               0
stock                   0
proveedor               1
precio_venta_publico    0
dtype: int64


In [1408]:
print((df1 == "").sum())
print((df2 == "").sum())
print((df3 == "").sum())

id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64


In [1409]:
print(df1['precio_compra'])
print(df2['precio_compra'])
print(df3['precio_compra'])

0      0.35
1       1.8
2       0.2
3     error
4      0.15
5       0.9
6      None
7      0.25
8       3.2
9       0.6
10      2.5
11      1.1
12     0.45
13      1.6
14      0.3
15      1.2
16      0.5
17      0.4
18      3.8
19     0.95
Name: precio_compra, dtype: object
0      0.15
1      1.40
2      2.10
3      0.55
4      0.60
5      0.30
6       NaN
7      5.20
8       NaN
9      4.50
10    18.00
11     3.10
12     0.70
13     0.50
14     1.20
15     0.80
16     2.80
17     1.10
18     2.40
19     0.35
Name: precio_compra, dtype: float64
0      1.50
1      2.80
2      3.60
3      0.90
4      0.70
5       NaN
6     12.50
7      1.80
8      1.80
9      1.40
10     0.55
11     4.20
12      NaN
13     2.90
14     0.35
15     0.40
16     0.80
17     0.65
18     1.30
19     2.60
Name: precio_compra, dtype: float64


In [1410]:
df1['stock'] = pd.to_numeric(df1['stock'], errors='coerce')
df2['stock'] = pd.to_numeric(df2['stock'], errors='coerce')
df3['stock'] = pd.to_numeric(df3['stock'], errors='coerce')
print(df1[df1['stock'] < 0])
print(df2[df2['stock'] < 0])
print(df3[df3['stock'] < 0])

    id                nombre precio_compra     categoria  stock  \
19  20  Crayones x12 Colores          0.95  Manualidades  -30.0   

        proveedor  precio_venta_publico  
19  Norma Ecuador                   1.8  
Empty DataFrame
Columns: [id, nombre, precio_compra, categoria, stock, proveedor, precio_venta_publico]
Index: []
    id                    nombre  precio_compra     categoria  stock  \
16  57  Cartón Corrugado 50x70cm            0.8  Manualidades  -15.0   

              proveedor precio_venta_publico  
16  PapelPro Cía. Ltda.                 1.60  


### A-6. Concatenar con `pd.concat()` y exportar con el metodo to_csv()

`pd.concat()` apila DataFrames verticalmente. `ignore_index=True` reinicia el índice.

| Función | Uso |
|---------|-----|
| `pd.concat()` | Apilar DataFrames con **misma estructura** |

In [1411]:
df_total = pd.concat([df1, df2, df3], ignore_index=True)
df_total

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,1,Bolígrafo BIC Azul,0.35,Escritura,500.0,DistribuPapel S.A.,0.75
1,2,Cuaderno Universitario 100 hojas,1.8,Cuadernos,300.0,Norma Ecuador,3.5
2,3,Lápiz HB Faber-Castell,0.2,Escritura,800.0,DistribuPapel S.A.,0.45
3,4,Resaltador Amarillo,error,Escritura,150.0,OfficeMax Ecuador,1.2
4,5,Carpeta Manila A4,0.15,Archivo,1000.0,PapelPro Cía. Ltda.,0.3
5,6,Tijeras Escolares 13cm,0.9,Manualidades,200.0,OfficeMax Ecuador,1.8
6,7,Corrector Líquido 18ml,None,Escritura,180.0,Norma Ecuador,1.5
7,8,Goma de Borrar Blanca,0.25,Escritura,600.0,DistribuPapel S.A.,0.55
8,9,Resma Papel Bond A4 75g,3.2,Papel,250.0,PapelPro Cía. Ltda.,5.5
9,10,Marcador Permanente Negro,0.6,Escritura,400.0,OfficeMax Ecuador,1.1


Exportar el dataframe concatenado con `to_csv()` con index=False  como argumento adicional, evitar escribir el índice numérico como columna extra

In [1412]:
df_total.to_csv("productos_final.csv", index=False)

---
---
## PARTE B: Series, DataFrames, `loc` e `iloc` — Catálogo de Netflix

Trabajaremos con el archivo `netflix.csv`, un dataset real con el catálogo de contenidos de Netflix.

| Columna | Descripción |
|---------|------------|
| `show_id` | ID único del título |
| `type` | `Movie` o `TV Show` |
| `title` | Nombre del título |
| `director` | Director(es) |
| `cast` | Elenco principal |
| `country` | País de producción |
| `date_added` | Fecha de añadido a Netflix |
| `release_year` | Año de estreno |
| `rating` | Clasificación de audiencia |
| `duration` | Duración (minutos o temporadas) |
| `listed_in` | Géneros/categorías |
| `description` | Sinopsis |

### B-1. Carga el dataset y realiza una primera inspección

In [1413]:
netflix=pd.read_csv("netflix.csv")
netflix.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.601,NaN
1,tm82169,Rocky,MOVIE,"When world heavyweight boxing champion, Apollo...",1976,PG,119,"['drama', 'sport']",['US'],NaN,tt0075148,8.1,588100.0,106.361,7.782
2,tm17823,Grease,MOVIE,Australian good girl Sandy and greaser Danny f...,1978,PG,110,"['romance', 'comedy']",['US'],NaN,tt0077631,7.2,283316.0,33.160,7.406
3,tm191099,The Sting,MOVIE,A novice con man teams up with an acknowledged...,1973,PG,129,"['crime', 'drama', 'comedy', 'music']",['US'],NaN,tt0070735,8.3,266738.0,24.616,8.020
4,tm69975,Rocky II,MOVIE,After Rocky goes the distance with champ Apoll...,1979,PG,119,"['drama', 'sport']",['US'],NaN,tt0079817,7.3,216307.0,75.699,7.246


---
### B-2. ¿Qué es una Series?

Una **Series** es un arreglo **unidimensional** etiquetado. Cada elemento tiene un **valor** y un **índice**.

```python
# Desde una lista
s = pd.Series([10, 20, 30])

# Desde un diccionario (claves = índice)
s = pd.Series({'a': 10, 'b': 20, 'c': 30})
```

| Atributo | Descripción |
|----------|------------|
| `.values` | Datos como array NumPy |
| `.index` | Etiquetas del índice |
| `.dtype` | Tipo de dato |
| `.name` | Nombre de la Series |
| `.shape` | Forma (número de elementos,) |

**Cuando accedes a UNA columna de un DataFrame, obtienes una Series:**

### Extraer una columna = una Series del csv de netflix

In [1414]:
columna=pd.Series(netflix['title'])
columna

0       Five Came Back: The Reference Films
1                                     Rocky
2                                    Grease
3                                 The Sting
4                                  Rocky II
                       ...                 
6132                          عبود في البيت
6133                                Sweetie
6134              Sommore: Queen Chandelier
6135                           All Na Vibes
6136                        Going to Heaven
Name: title, Length: 6137, dtype: str

### Operaciones comunes sobre Series

### Obten las estadísticas de la columna 'release_year', count, min, max, mean, median, std

In [1415]:
columna2=pd.Series(netflix['release_year'])
print(columna2.value_counts())
print("------------------------------")
print(columna2.min())
print("------------------------------")
print(columna2.max())
print("------------------------------")
print(columna2.mean())
print("------------------------------")
print(columna2.median())
print("------------------------------")
print(columna2.std())
print("------------------------------")


release_year
2022    915
2021    868
2020    829
2019    788
2018    743
       ... 
1974      1
1962      1
1963      1
1956      1
1970      1
Name: count, Length: 62, dtype: int64
------------------------------
1945
------------------------------
2023
------------------------------
2017.3718429199935
------------------------------
2019.0
------------------------------
6.603620140967784
------------------------------


### Obten el Top 10 países con más contenido producido

In [1416]:
top_10=pd.Series(netflix['production_countries']).value_counts().head(10)
top_10

production_countries
['US']    1981
['IN']     633
['JP']     278
['KR']     259
['GB']     235
['ES']     177
[]         176
['FR']     121
['MX']     115
['BR']     104
Name: count, dtype: int64

### Obten el Top 10 de series/peliculas mas vistas

In [1417]:
top_10_populares=pd.Series(netflix['tmdb_popularity']).value_counts().head(10)
top_10_populares

tmdb_popularity
1.400    24
0.600    22
1.960     7
3.412     5
2.672     5
2.346     4
3.115     4
2.370     4
3.007     4
2.154     4
Name: count, dtype: int64

---
### B-3. Acceso con `loc` (por etiqueta)

**`loc`** selecciona datos por **etiqueta** del índice y **nombre** de columna.

```python
df.loc[etiqueta_fila]                    # Una fila completa
df.loc[etiqueta_fila, 'columna']         # Un valor específico
df.loc[inicio:fin]                       # Rango de filas (INCLUSIVO)
df.loc[condicion]                        # Filtrado booleano
df.loc[condicion, ['col1', 'col2']]      # Filtrado + columnas
```

> Con `loc`, los rangos son **INCLUSIVOS** en ambos extremos: `loc[2:5]` incluye 2, 3, 4 **y 5**.

In [1418]:
# Genera un ejemplo de Acceso a una fila completa por su etiqueta de índice
fila_completa = netflix.loc[1]
print(fila_completa)


id                                                                tm82169
title                                                               Rocky
type                                                                MOVIE
description             When world heavyweight boxing champion, Apollo...
release_year                                                         1976
age_certification                                                      PG
runtime                                                               119
genres                                                 ['drama', 'sport']
production_countries                                               ['US']
seasons                                                               NaN
imdb_id                                                         tt0075148
imdb_score                                                            8.1
imdb_votes                                                       588100.0
tmdb_popularity                       

In [1419]:
# Filtrado con MÚLTIPLES condiciones
# Obten el listado de Películas de Estados Unidos producidas desde 2018
netflix.loc[(netflix.production_countries == "['US']") & (netflix.release_year >= 2018)]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
1422,ts79813,You,SHOW,"A dangerously charming, intensely obsessive yo...",2018,TV-MA,48,"['drama', 'romance', 'thriller', 'crime']",['US'],4.0,tt7335184,7.7,261831.0,341.151,8.118
1423,ts81927,New Amsterdam,SHOW,The new medical director breaks the rules to h...,2018,TV-14,43,['drama'],['US'],5.0,tt7817340,8.0,43464.0,163.879,8.462
1428,ts78801,Cobra Kai,SHOW,This Karate Kid sequel series picks up 30 year...,2018,TV-14,33,"['action', 'drama', 'comedy', 'sport']",['US'],6.0,tt7221388,8.5,190785.0,218.009,8.200
1431,ts81294,Manifest,SHOW,After landing from a turbulent but routine fli...,2018,TV-14,42,"['scifi', 'drama', 'thriller']",['US'],4.0,tt8421350,7.1,71203.0,176.155,7.733
1432,ts83970,All American,SHOW,When a rising high school football player from...,2018,TV-14,43,"['drama', 'sport']",['US'],5.0,tt11574984,7.6,9704.0,75.706,8.209
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6098,ts308560,African Queens: Njinga,SHOW,Expert interviews and other documentary conten...,2023,TV-14,47,"['history', 'documentation']",['US'],2.0,tt15305648,5.5,361.0,6.023,NaN
6109,ts367818,Bling Empire: New York,SHOW,"Meet a fresh group of wealthy, sophisticated a...",2023,TV-MA,40,['reality'],['US'],1.0,tt22481904,5.2,376.0,6.621,NaN
6122,ts364674,Princess Power,SHOW,Princess friends from four different Fruitdoms...,2023,TV-Y,15,"['family', 'fantasy', 'animation', 'comedy']",['US'],1.0,tt22013036,7.0,60.0,6.319,8.500
6126,tm1291873,Andrew Santino: Cheeseburger,MOVIE,No topic is safe in this unfiltered stand-up s...,2023,NaN,58,"['comedy', 'documentation']",['US'],NaN,tt24640480,6.5,808.0,3.181,5.500


In [1420]:
# Filtrado con isin() para múltiples valores
# Filtra todas las filas cuyo Contenido tenga como origen Latinoamérica
paises_latam = ["['MX']", "['AR']", "['CO']", "['CL']", "['BR']", "['PE']"]
netflix.loc[netflix.production_countries.isin(paises_latam)]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
63,tm126769,Waiting for the Hearse,MOVIE,"Mama Cora, who is almost eighty years old, has...",1985,NaN,87,['comedy'],['AR'],NaN,tt0089108,8.0,6714.0,7.593,8.180
127,ts28516,Okupas,SHOW,"During the year 2000, Ricardo, Pollo, Walter a...",2000,TV-MA,40,"['drama', 'crime']",['AR'],1.0,tt0289649,9.0,2630.0,7.290,8.635
136,tm41525,Herod's Law,MOVIE,"Mexico, 1949. The fable of a janitor turned Ma...",2000,R,123,"['comedy', 'crime', 'drama']",['MX'],NaN,tt0221344,7.8,6414.0,12.915,7.983
155,ts224786,Escalona,SHOW,"The improbable real life of Rafael Escalona, w...",1991,TV-MA,44,['drama'],['CO'],1.0,NaN,NaN,NaN,8.981,7.612
247,ts38488,Hidden Passion,SHOW,The Reyes-Elizondo's idyllic lives are shatter...,2003,TV-14,43,"['drama', 'romance']",['CO'],2.0,tt0387763,7.8,2632.0,164.924,7.644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6102,ts321003,The Endless Night,SHOW,"A fire at Nightclub Kiss, which killed 242 peo...",2023,TV-MA,43,"['crime', 'drama', 'thriller']",['BR'],1.0,tt16427924,7.7,1124.0,13.943,8.200
6103,ts374551,Against the Ropes,SHOW,They say being a woman is all a business and m...,2023,TV-MA,40,"['drama', 'comedy', 'sport']",['MX'],1.0,tt25031242,6.5,207.0,29.503,9.083
6110,tm1301192,Whindersson Nunes: Preaching to the Choir,MOVIE,The new show by comedian Whindersson Nunes ent...,2023,NaN,67,['comedy'],['BR'],NaN,tt26440342,6.4,182.0,11.810,7.200
6114,tm1297263,All the Places,MOVIE,An estranged brother and sister reunite at the...,2023,NaN,97,"['comedy', 'drama']",['MX'],NaN,tt12616964,5.6,249.0,292.518,6.800


In [1421]:
# str.contains(): filtrado por texto parcial en columna
# Títulos que contengan 'Love' en el nombre
netflix.loc[netflix.title.str.contains("Love")]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
299,tm97072,Eat Pray Love,MOVIE,Liz Gilbert had everything a modern woman is s...,2010,PG-13,133,"['drama', 'romance']",['US'],NaN,tt0879870,5.8,100026.0,19.052,6.230
322,tm464888,Love Aaj Kal,MOVIE,When professional ambitions clash with persona...,2009,PG-13,141,"['drama', 'comedy', 'romance']",['IN'],NaN,tt1275863,6.8,16443.0,3.097,5.500
418,ts4866,Fated to Love You,SHOW,"Fated to Love You, also known as You're My Des...",2008,TV-G,67,"['drama', 'comedy', 'romance']",['TW'],1.0,tt1483930,7.4,510.0,16.052,6.833
441,tm95434,Love in a Puff,MOVIE,When the Hong Kong government enacts a ban on ...,2010,NaN,104,"['comedy', 'drama', 'romance']",['HK'],NaN,tt1602479,7.1,2746.0,4.220,6.953
447,tm76399,A Love Story,MOVIE,Ian Montes is a picture of success. Despite be...,2007,NaN,117,"['romance', 'drama']",['PH'],NaN,tt0990433,6.5,150.0,1.410,4.600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6076,ts360024,In Love All Over Again,SHOW,"Year 2003, Irene a young film student who is p...",2023,TV-MA,45,"['drama', 'romance', 'comedy']",['ES'],1.0,tt21206964,6.3,669.0,85.435,6.400
6078,tm1304302,Love at First Kiss,MOVIE,"The story of Javier who, at the age of 16, whi...",2023,NaN,96,"['comedy', 'romance']",['ES'],NaN,tt14463514,5.7,897.0,152.671,6.246
6100,tm1300541,Squared Love All Over Again,MOVIE,The relationship of a well-known journalist an...,2023,NaN,99,"['comedy', 'romance']",['PL'],NaN,tt26369131,4.5,383.0,25.780,5.600
6101,ts314687,Love to Hate You,SHOW,"For a woman who despises losing to men, and a ...",2023,TV-MA,54,"['comedy', 'drama', 'romance']",['KR'],1.0,tt26229508,7.9,1584.0,36.117,8.300


### B-4. Ya que tienes acceso a los datos de Netflix, en base a tu curiosidad, usando pandas obten al menos 3 datos de interes adicionales, que te gustaria conocer.

In [1422]:
netflix.loc[(netflix.release_year < 2000)]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.601,NaN
1,tm82169,Rocky,MOVIE,"When world heavyweight boxing champion, Apollo...",1976,PG,119,"['drama', 'sport']",['US'],NaN,tt0075148,8.1,588100.0,106.361,7.782
2,tm17823,Grease,MOVIE,Australian good girl Sandy and greaser Danny f...,1978,PG,110,"['romance', 'comedy']",['US'],NaN,tt0077631,7.2,283316.0,33.160,7.406
3,tm191099,The Sting,MOVIE,A novice con man teams up with an acknowledged...,1973,PG,129,"['crime', 'drama', 'comedy', 'music']",['US'],NaN,tt0070735,8.3,266738.0,24.616,8.020
4,tm69975,Rocky II,MOVIE,After Rocky goes the distance with champ Apoll...,1979,PG,119,"['drama', 'sport']",['US'],NaN,tt0079817,7.3,216307.0,75.699,7.246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,tm204173,Aashik Aawara,MOVIE,"Raised by a kindly thief, orphaned Jimmy goes ...",1993,NaN,153,"['romance', 'comedy', 'drama']",['IN'],NaN,tt0106206,4.8,345.0,1.462,5.500
182,tm1096916,The Trial of Adolf Eichmann,MOVIE,The 1961 trial of Adolf Eichmann held in an Is...,1997,NaN,90,['documentation'],['FR'],NaN,tt0280153,7.4,123.0,1.448,7.700
183,tm326102,Qila,MOVIE,"When an abusive landowner is murdered, his twi...",1998,NaN,160,"['drama', 'family', 'music']",['IN'],NaN,tt0286907,4.9,137.0,1.414,6.000
184,tm7810,Nightmare in Columbia County,MOVIE,The recounting of a terrible crime that wracke...,1991,NaN,100,"['drama', 'crime', 'thriller']",['US'],NaN,tt0102537,5.5,364.0,1.357,6.000


In [1423]:
netflix.loc[netflix.type.str.contains("MOVIE")]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
1,tm82169,Rocky,MOVIE,"When world heavyweight boxing champion, Apollo...",1976,PG,119,"['drama', 'sport']",['US'],NaN,tt0075148,8.1,588100.0,106.361,7.782
2,tm17823,Grease,MOVIE,Australian good girl Sandy and greaser Danny f...,1978,PG,110,"['romance', 'comedy']",['US'],NaN,tt0077631,7.2,283316.0,33.160,7.406
3,tm191099,The Sting,MOVIE,A novice con man teams up with an acknowledged...,1973,PG,129,"['crime', 'drama', 'comedy', 'music']",['US'],NaN,tt0070735,8.3,266738.0,24.616,8.020
4,tm69975,Rocky II,MOVIE,After Rocky goes the distance with champ Apoll...,1979,PG,119,"['drama', 'sport']",['US'],NaN,tt0079817,7.3,216307.0,75.699,7.246
5,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'comedy']",['GB'],NaN,tt0071853,8.2,547292.0,20.964,7.804
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6132,tm1303784,عبود في البيت,MOVIE,Two young boys must work together to stop robb...,2023,NaN,81,"['family', 'comedy']",['KW'],NaN,NaN,NaN,NaN,3.351,2.000
6133,tm1260999,Sweetie,MOVIE,"‘Theatre is my life,’ Yıldız Kenter admits in ...",2023,NaN,120,['documentation'],['TR'],NaN,tt26349328,7.9,209.0,2.450,7.200
6134,tm1310286,Sommore: Queen Chandelier,MOVIE,This Queen of Comedy shines as she takes the s...,2023,NaN,69,['comedy'],['US'],NaN,tt21033382,6.1,91.0,1.960,NaN
6135,tm1072700,All Na Vibes,MOVIE,The lives of three teenagers and a hit-man int...,2023,NaN,80,['drama'],['NG'],NaN,tt14922926,5.2,18.0,1.357,4.000


In [1424]:
netflix.loc[netflix.genres.str.contains("comedy")]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
2,tm17823,Grease,MOVIE,Australian good girl Sandy and greaser Danny f...,1978,PG,110,"['romance', 'comedy']",['US'],NaN,tt0077631,7.2,283316.0,33.160,7.406
3,tm191099,The Sting,MOVIE,A novice con man teams up with an acknowledged...,1973,PG,129,"['crime', 'drama', 'comedy', 'music']",['US'],NaN,tt0070735,8.3,266738.0,24.616,8.020
5,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'comedy']",['GB'],NaN,tt0071853,8.2,547292.0,20.964,7.804
6,tm17249,Animal House,MOVIE,"At a 1962 College, Dean Vernon Wormer is deter...",1978,R,109,['comedy'],['US'],NaN,tt0077975,7.4,123611.0,17.372,7.020
7,ts22164,Monty Python's Flying Circus,SHOW,A British sketch comedy series with the shows ...,1969,TV-14,30,"['comedy', 'european']",['GB'],4.0,tt0063929,8.8,75654.0,24.773,8.258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6128,tm1307029,Married to Work,MOVIE,"To save their real estate agency, an ambitious...",2023,NaN,76,"['comedy', 'romance']","['KE', 'TZ', 'NG']",NaN,tt26687699,7.5,25.0,8.446,3.800
6130,tm1299701,Dr Jason Leong: Ride with Caution,MOVIE,"The comic shares his diagnoses on ageing, the ...",2023,NaN,65,['comedy'],['MY'],NaN,tt26340263,5.9,102.0,5.379,2.000
6131,tm1315251,Do Your Worst,MOVIE,Regret and redemption take center stage as Son...,2023,NaN,91,"['drama', 'romance', 'comedy']",['ZA'],NaN,tt27032785,NaN,NaN,3.842,NaN
6132,tm1303784,عبود في البيت,MOVIE,Two young boys must work together to stop robb...,2023,NaN,81,"['family', 'comedy']",['KW'],NaN,NaN,NaN,NaN,3.351,2.000


---
### B-5. Acceso con `iloc` (por posición entera)

**`iloc`** selecciona datos por **posición numérica** (como índices de listas en Python).

```python
df.iloc[posicion]                          # Una fila
df.iloc[pos_fila, pos_columna]             # Un valor
df.iloc[inicio:fin]                        # Rango (EXCLUSIVO al final)
df.iloc[[0, 5, 10]]                        # Posiciones específicas
df.iloc[::n]                               # Cada n filas
```

> Con `iloc`, el rango es **EXCLUSIVO** al final: `iloc[2:5]` incluye posiciones 2, 3, 4 pero **NO 5**.

### Diferencia clave `loc` vs `iloc`:

| | `loc` | `iloc` |
|--|-------|--------|
| **Acceso** | Por **etiqueta** | Por **posición** |
| **Rango** | **Inclusivo** | **Exclusivo** |
| **Columnas** | Por nombre | Por número (0, 1, 2...) |
| **Filtrado** |  Booleano | No directo |

In [1425]:
# Obten la Primera fila (posición 0) usando loc
netflix.iloc[0]

id                                                               ts300399
title                                 Five Came Back: The Reference Films
type                                                                 SHOW
description             This collection includes 12 World War II-era p...
release_year                                                         1945
age_certification                                                   TV-MA
runtime                                                                51
genres                                                  ['documentation']
production_countries                                               ['US']
seasons                                                               1.0
imdb_id                                                               NaN
imdb_score                                                            NaN
imdb_votes                                                            NaN
tmdb_popularity                       

In [1426]:
# Última fila (índice negativo, como listas de Python) usando loc
netflix.iloc[-1]

id                                                               tm561709
title                                                     Going to Heaven
type                                                                MOVIE
description             A story about young boy sultan, 11 years old l...
release_year                                                         2023
age_certification                                                     NaN
runtime                                                                90
genres                                                         ['family']
production_countries                                               ['AE']
seasons                                                               NaN
imdb_id                                                         tt4623222
imdb_score                                                            7.0
imdb_votes                                                           40.0
tmdb_popularity                       

In [1427]:
# Valor específico: fila 10, columna 2 (title) usando loc
netflix.iloc[10, 2]

'MOVIE'

In [1428]:
# Rango de filas (EXCLUSIVO: posiciones 2, 3, 4 — NO incluye 5) usando loc
netflix.iloc[[2, 3, 4]]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
2,tm17823,Grease,MOVIE,Australian good girl Sandy and greaser Danny f...,1978,PG,110,"['romance', 'comedy']",['US'],NaN,tt0077631,7.2,283316.0,33.160,7.406
3,tm191099,The Sting,MOVIE,A novice con man teams up with an acknowledged...,1973,PG,129,"['crime', 'drama', 'comedy', 'music']",['US'],NaN,tt0070735,8.3,266738.0,24.616,8.020
4,tm69975,Rocky II,MOVIE,After Rocky goes the distance with champ Apoll...,1979,PG,119,"['drama', 'sport']",['US'],NaN,tt0079817,7.3,216307.0,75.699,7.246


In [1429]:
# Usando loc , obten los 10 títulos más antiguos en Netflix
# Combinamos sort_values con iloc(Ejemplo base: netflix_ordenado = df_netflix.sort_values("release_year", ascending=True))
print('=== 10 títulos más antiguos en Netflix ===')
netflixOrdenado = netflix.sort_values("release_year", ascending=True)
top10Antiguos = netflixOrdenado.loc[:, ["title", "release_year"]].head(10)
print(top10Antiguos)

=== 10 títulos más antiguos en Netflix ===
                                  title  release_year
0   Five Came Back: The Reference Films          1945
27                      The Blazing Sun          1954
9                       White Christmas          1954
26                          Dark Waters          1956
12                        Cairo Station          1958
22                            Professor          1962
24               Saladin the Victorious          1963
19                             Amrapali          1966
15                               Prince          1969
7          Monty Python's Flying Circus          1969


---
---
## PARTE C: Procesamiento de Texto — *Dracula* (Bram Stoker)

En esta sección crear las cajas pertinentes a fin de realizar las siguientes actividades sobre el libro de Drácula:
1. Descargar texto desde una URL con `urllib`, crear una variable time, que contabilize el tiempo desde que se inicia la descarga.
2. Eliminar encabezados y pies de página propios de Project Gutenberg.
3. Eliminar signos de puntuación, números y caracteres especiales con `re`.
4. Procesar texto: minúsculas, tokenizar, filtrar.
5. Contar frecuencias con `collections.Counter`.
6. Mostrar el top 20 de las palabras mas frecuentes.
7. Guardar resultados en un archivo de texto(txt), donde se muestre el top 20 de palabras mas frecuentes y su cantidad de incidencias, asi como el tiempo total demorado desde que se descarga el libro, hasta que se cuenta las palabras mas frecuentes.

Usaremos **"Dracula"** de Bram Stoker, disponible en [Project Gutenberg](https://www.gutenberg.org/ebooks/345).  
https://www.gutenberg.org/cache/epub/345/pg345.txt



In [1430]:
#Desarrollo del estudiante
inicio = time.time()

url = "https://www.gutenberg.org/cache/epub/345/pg345.txt"

with urllib.request.urlopen(url) as response:
    texto = response.read().decode("utf-8")

inicio_texto = texto.find("*** START OF THE PROJECT GUTENBERG EBOOK")
fin_texto = texto.find("*** END OF THE PROJECT GUTENBERG EBOOK")
texto = texto[inicio_texto:fin_texto]

texto = re.sub(r'[^a-zA-Z\s]', '', texto)

texto = texto.lower()                  
palabras = texto.split()               

palabras = [p for p in palabras if len(p) > 2]

conteo = Counter(palabras)

top20 = conteo.most_common(20)

fin = time.time()
tiempo_total = fin - inicio

print("=== TOP 20 PALABRAS ===")
for palabra, freq in top20:
    print(palabra, ":", freq)

print("\nTiempo total:", tiempo_total, "segundos")

with open("resultado_dracula.txt", "w", encoding="utf-8") as f:
    f.write("=== TOP 20 PALABRAS ===\n")
    for palabra, freq in top20:
        f.write(f"{palabra}: {freq}\n")
    
    f.write(f"\nTiempo total: {tiempo_total} segundos")

=== TOP 20 PALABRAS ===
the : 7861
and : 5827
that : 2439
was : 1874
for : 1509
his : 1467
not : 1381
you : 1381
with : 1274
all : 1150
but : 1065
have : 1058
her : 1056
had : 1038
him : 941
she : 809
when : 771
there : 766
which : 649
from : 618

Tiempo total: 5.215266227722168 segundos
